# Fraud Transaction Risk Intelligence
## 03 — Feature Engineering

Transform the prepared transaction data into informative, model-ready features for fraud prediction.

### Pipeline Stage 3 — Feature Engineering

Build predictive features from transaction, time, payment, device and missing-value information identified during EDA.

In [2]:
import pandas as pd
import numpy as np
import os

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)

## 1. Load Prepared Data

Load the cleaned transaction and identity dataset produced in Notebook 1.

In [3]:
DATA_PATH = "../data/processed/train_prepared.parquet"

train = pd.read_parquet(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", train.shape)

Dataset loaded successfully!
Shape: (590540, 434)


## 2. Feature Overview

Review available variables before creating new features.

In [4]:
print("Total features:", len(train.columns))
print("\nData types:")
print(train.dtypes.value_counts())

Total features: 434

Data types:
float32    399
str         31
int32        2
int8         1
int16        1
Name: count, dtype: int64


## 3. Transaction Time Features

Convert `TransactionDT` into interpretable time-based features that may capture changes in fraud behaviour.

In [5]:
SECONDS_PER_DAY = 24 * 60 * 60

train["TransactionHour"] = (
    (train["TransactionDT"] // 3600) % 24
).astype("int8")

train["TransactionDay"] = (
    train["TransactionDT"] // SECONDS_PER_DAY
).astype("int32")

train["TransactionWeek"] = (
    train["TransactionDay"] // 7
).astype("int32")

In [6]:
train[
    [
        "TransactionDT",
        "TransactionHour",
        "TransactionDay",
        "TransactionWeek"
    ]
].head()

,TransactionDT,TransactionHour,TransactionDay,TransactionWeek
0,86400,0,1,0
1,86401,0,1,0
2,86469,0,1,0
3,86499,0,1,0
4,86506,0,1,0


## 4. Transaction Amount Features

Create transformed amount features to capture scale and skewness in transaction values.

In [7]:
train["LogTransactionAmt"] = np.log1p(
    train["TransactionAmt"].clip(lower=0)
)

train["TransactionAmtDecimals"] = (
    (train["TransactionAmt"] * 100) % 100
).round()

In [8]:
train[
    [
        "TransactionAmt",
        "LogTransactionAmt",
        "TransactionAmtDecimals"
    ]
].describe()

,TransactionAmt,LogTransactionAmt,TransactionAmtDecimals
count,590540.000000,590540.000000,590540.000000
mean,135.027176,4.382960,37.945118
std,239.162521,0.937183,43.412449
min,0.251000,0.223943,0.000000
25%,43.320999,3.791459,0.000000
50%,68.769001,4.245190,0.000000
75%,125.000000,4.836282,95.000000
max,31937.390625,10.371564,100.000000


## 5. Relative Transaction Amount

Compare each transaction amount with the overall transaction distribution.

In [9]:
amount_median = train["TransactionAmt"].median()
amount_std = train["TransactionAmt"].std()

train["AmountVsMedian"] = (
    train["TransactionAmt"] / amount_median
)

train["AmountZScore"] = (
    (train["TransactionAmt"] - amount_median) / amount_std
)

## 6. Missingness Features

Missing information can itself contain useful fraud signals, particularly for identity and device attributes.

In [10]:
train["MissingFeatureCount"] = train.isnull().sum(axis=1)

train["MissingFeatureRatio"] = (
    train["MissingFeatureCount"] / train.shape[1]
)

In [11]:
train[
    [
        "MissingFeatureCount",
        "MissingFeatureRatio"
    ]
].describe()

,MissingFeatureCount,MissingFeatureRatio
count,590540.000000,590540.000000
mean,195.622774,0.442585
std,49.035963,0.110941
min,24.000000,0.054299
25%,208.000000,0.470588
50%,211.000000,0.477376
75%,229.000000,0.518100
max,340.000000,0.769231


## 7. Identity Features

Capture the availability and basic structure of device and identity information.

In [12]:
identity_cols = [
    col for col in train.columns
    if col.startswith("id_")
]

print("Identity features:", len(identity_cols))

Identity features: 38


In [13]:
train["IdentityFeatureCount"] = (
    train[identity_cols].notnull().sum(axis=1)
)

train["HasIdentityInfo"] = (
    train["IdentityFeatureCount"] > 0
).astype("int8")

## 8. Device Features

Create simple indicators from available device information.

In [14]:
if "DeviceInfo" in train.columns:
    train["HasDeviceInfo"] = (
        train["DeviceInfo"].notnull()
    ).astype("int8")
else:
    train["HasDeviceInfo"] = 0

## 9. Card and Payment Features

Create indicators for the availability of card-related information.

In [15]:
card_cols = [
    col for col in train.columns
    if col.startswith("card")
]

print("Card-related features:", len(card_cols))

Card-related features: 6


In [16]:
train["CardFeatureCount"] = (
    train[card_cols].notnull().sum(axis=1)
)

## 10. Email Features

Create indicators for purchaser and recipient email information.

In [17]:
email_cols = [
    col for col in ["P_emaildomain", "R_emaildomain"]
    if col in train.columns
]

for col in email_cols:
    train[f"{col}_Missing"] = (
        train[col].isnull()
    ).astype("int8")

## 11. Address Features

Capture the availability of address-related transaction information.

In [18]:
address_cols = [
    col for col in ["addr1", "addr2"]
    if col in train.columns
]

if address_cols:
    train["AddressFeatureCount"] = (
        train[address_cols].notnull().sum(axis=1)
    )
else:
    train["AddressFeatureCount"] = 0

## 12. Categorical Features

Convert selected categorical variables into numerical representations for machine learning.

In [19]:
categorical_candidates = [
    "ProductCD",
    "card4",
    "card6",
    "DeviceType"
]

categorical_cols = [
    col for col in categorical_candidates
    if col in train.columns
]

categorical_cols

['ProductCD', 'card4', 'card6', 'DeviceType']

In [20]:
for col in categorical_cols:
    train[col] = train[col].fillna("Unknown").astype("category")

print(train[categorical_cols].dtypes)

ProductCD     category
card4         category
card6         category
DeviceType    category
dtype: object


## 13. Frequency Encoding

Represent categorical values by how frequently they occur in the dataset.

In [21]:
frequency_cols = [
    col for col in [
        "ProductCD",
        "card4",
        "card6",
        "DeviceType"
    ]
    if col in train.columns
]

for col in frequency_cols:
    freq = train[col].value_counts(dropna=False)
    train[f"{col}_freq"] = train[col].map(freq).astype("float32")

## 14. Interaction Features

Create a small number of interaction features to capture relationships between transaction attributes.

In [22]:
if "TransactionAmt" in train.columns and "ProductCD" in train.columns:
    train["Amount_Product_Interaction"] = (
        train["TransactionAmt"] *
        train["ProductCD"].cat.codes
    )

## 15. Feature Validation

Check the resulting feature set and identify any remaining issues.

In [23]:
print("Feature count:", len(train.columns))
print("Dataset shape:", train.shape)

print("\nMissing target values:")
print(train["isFraud"].isnull().sum())

print("\nDuplicate rows:")
print(train.duplicated().sum())

Feature count: 455
Dataset shape: (590540, 455)

Missing target values:
0

Duplicate rows:
0


## 16. Model Input Check

Identify categorical and non-numeric columns that will require encoding or removal before modelling.

In [24]:
non_numeric_cols = train.select_dtypes(
    exclude=np.number
).columns.tolist()

print("Non-numeric features:", len(non_numeric_cols))
print(non_numeric_cols)

Non-numeric features: 31
['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']


## 17. Save Engineered Dataset

Save the engineered feature set for the modelling stage.

In [25]:
OUTPUT_PATH = "../data/processed"

os.makedirs(OUTPUT_PATH, exist_ok=True)

train.to_parquet(
    os.path.join(OUTPUT_PATH, "train_featured.parquet"),
    index=False
)

print("Engineered dataset saved successfully!")

Engineered dataset saved successfully!


## 18. Feature Engineering Summary

The prepared dataset has been extended with time, amount, missingness, identity, payment and frequency-based features for fraud modelling.

In [26]:
print("Final dataset shape:", train.shape)

print("\nNew engineered features:")
engineered_features = [
    "TransactionHour",
    "TransactionDay",
    "TransactionWeek",
    "LogTransactionAmt",
    "TransactionAmtDecimals",
    "AmountVsMedian",
    "AmountZScore",
    "MissingFeatureCount",
    "MissingFeatureRatio",
    "IdentityFeatureCount",
    "HasIdentityInfo",
    "HasDeviceInfo",
    "CardFeatureCount",
    "AddressFeatureCount"
]

for feature in engineered_features:
    if feature in train.columns:
        print("-", feature)

Final dataset shape: (590540, 455)

New engineered features:
- TransactionHour
- TransactionDay
- TransactionWeek
- LogTransactionAmt
- TransactionAmtDecimals
- AmountVsMedian
- AmountZScore
- MissingFeatureCount
- MissingFeatureRatio
- IdentityFeatureCount
- HasIdentityInfo
- HasDeviceInfo
- CardFeatureCount
- AddressFeatureCount


# Conclusion

The feature engineering stage transformed raw transaction information into additional time, amount, missingness, identity, payment and frequency-based signals.

The resulting dataset is ready for fraud model development.